In [2]:
qtd_fornecedores = 200
meses = 24
tx_anomalia_mes = 0.06
seed = 42

In [3]:
import pandas as pd
import numpy as np
import random

random.seed(seed)
np.random.seed(seed)

In [4]:
gp_crescente = ['crescente'] * 80
gp_decrescente = ['decrescente'] * 60
gp_estavel = ['estavel'] * 60

grupos = gp_crescente + gp_decrescente + gp_estavel
random.shuffle(grupos)

In [5]:
fornecedores = []

for i in range(200):
    opcao = grupos[i]
    padroes = ['NUM', 'ACC', 'FORN']

    padrao_atual = random.choice(padroes)

    if padrao_atual == 'NUM':
        id_fornecedor = f'{1000000+i}'
    
    elif padrao_atual == 'ACC':
        id_fornecedor = f'ACC-{i+1:05d}'

    elif padrao_atual == 'FORN':
        id_fornecedor = f'FORN-{i+1:05d}'
    
    valor_medio_inicial = round(np.random.lognormal(8, 0.6), 2)

    if valor_medio_inicial <= 4000:
        freq_media_inicial = np.random.randint(10, 19)
    elif valor_medio_inicial > 4000 and valor_medio_inicial <= 10000:
        freq_media_inicial = np.random.randint(6, 13)
    elif valor_medio_inicial > 10000:
        freq_media_inicial = np.random.randint(3, 9)

    prob_inicio = np.random.uniform(0, 1)

    if prob_inicio < 0.6:
        mes_inicio = np.random.randint(1, 4)
    else:
        mes_inicio = np.random.randint(4, 24)

    if mes_inicio > 18:
        mes_fim = 24
    else:
        prob_fim = np.random.uniform(0, 1)
        if prob_fim < 0.6:
            mes_fim = 24
        else:
            mes_fim = np.random.randint(mes_inicio + 6, 25)

    fornecedores.append({
        'id_fornecedor_raw': id_fornecedor,
        'grupo': opcao,
        'valor_medio_inicial': valor_medio_inicial,
        'freq_media_inicial': freq_media_inicial,
        'mes_inicio': mes_inicio,
        'mes_fim': mes_fim
    })
df_fornecedores = pd.DataFrame(fornecedores)

print(mes_inicio, mes_fim)

#print(df_fornecedores)

3 19


In [6]:
df_meses = pd.DataFrame({'mes_pagamento': pd.RangeIndex(start=1, stop=25, step=1)})

df_fornecedor_mes_completo = df_fornecedores.merge(df_meses, how='cross')

df_fornecedor_mes_ativo = df_fornecedor_mes_completo.loc[
    (df_fornecedor_mes_completo['mes_pagamento']>=df_fornecedor_mes_completo['mes_inicio']) & 
    (df_fornecedor_mes_completo['mes_pagamento']<=df_fornecedor_mes_completo['mes_fim'])
]

display(df_fornecedor_mes_ativo)

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento
9,1000000,crescente,4015.95,8,10,24,10
10,1000000,crescente,4015.95,8,10,24,11
11,1000000,crescente,4015.95,8,10,24,12
12,1000000,crescente,4015.95,8,10,24,13
13,1000000,crescente,4015.95,8,10,24,14
...,...,...,...,...,...,...,...
4790,1000199,estavel,2483.11,13,3,19,15
4791,1000199,estavel,2483.11,13,3,19,16
4792,1000199,estavel,2483.11,13,3,19,17
4793,1000199,estavel,2483.11,13,3,19,18


In [7]:
taxa_freq = 0.015

min_lambda = 1.0
df_fornecedor_mes_ativo['mes_relativo'] = df_fornecedor_mes_ativo['mes_pagamento'] - df_fornecedor_mes_ativo['mes_inicio']
df_fornecedor_mes_ativo['lambda_mes'] = df_fornecedor_mes_ativo['freq_media_inicial']
#print(df_fornecedor_mes_ativo)

mask_crescente = df_fornecedor_mes_ativo['grupo'] == 'crescente'
#display(df_fornecedor_mes_ativo)
#display(mask_crescente)
df_fornecedor_mes_ativo['lambda_mes'] = df_fornecedor_mes_ativo.lambda_mes.astype(float)

# Crescente
df_fornecedor_mes_ativo.loc[ 
    mask_crescente, 
    'lambda_mes' 
    ] = df_fornecedor_mes_ativo['freq_media_inicial'] * (1 + taxa_freq) ** df_fornecedor_mes_ativo['mes_relativo']


mask_decrescente = df_fornecedor_mes_ativo['grupo'] == 'decrescente'
# Decrescente
df_fornecedor_mes_ativo.loc[ 
    mask_decrescente, 
    'lambda_mes' 
    ] = (df_fornecedor_mes_ativo['freq_media_inicial'] * (1 - taxa_freq) ** df_fornecedor_mes_ativo['mes_relativo']).clip(lower=min_lambda)

# display(df_fornecedor_mes_ativo[df_fornecedor_mes_ativo['id_fornecedor_raw'] == 'FORN-00003'])



C:\Users\Adam\AppData\Local\Temp\ipykernel_20848\1674474506.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fornecedor_mes_ativo['mes_relativo'] = df_fornecedor_mes_ativo['mes_pagamento'] - df_fornecedor_mes_ativo['mes_inicio']
C:\Users\Adam\AppData\Local\Temp\ipykernel_20848\1674474506.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fornecedor_mes_ativo['lambda_mes'] = df_fornecedor_mes_ativo['freq_media_inicial']
C:\Users\Adam\AppData\Local\Temp\ipykernel_20848\1674474506.py:11: SettingWithC

In [8]:
taxa_valor = 0.005

df_fornecedor_mes_ativo["quantidade_real_pagamentos"] = np.random.poisson(df_fornecedor_mes_ativo["lambda_mes"])

# Crescente
df_fornecedor_mes_ativo.loc[ 
    mask_crescente, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial'] * (1 + taxa_valor) ** df_fornecedor_mes_ativo['mes_relativo']

# Decrescente
df_fornecedor_mes_ativo.loc[ 
    mask_decrescente, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial'] * (1 - taxa_valor) ** df_fornecedor_mes_ativo['mes_relativo']

# Estável
mask_estavel = df_fornecedor_mes_ativo['grupo'] == 'estavel'
df_fornecedor_mes_ativo.loc[ 
    mask_estavel, 
    'valor_medio_mes' 
    ] = df_fornecedor_mes_ativo['valor_medio_inicial']

display(df_fornecedor_mes_ativo)
#display(df_fornecedor_mes_ativo)

C:\Users\Adam\AppData\Local\Temp\ipykernel_20848\1288299827.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fornecedor_mes_ativo["quantidade_real_pagamentos"] = np.random.poisson(df_fornecedor_mes_ativo["lambda_mes"])
C:\Users\Adam\AppData\Local\Temp\ipykernel_20848\1288299827.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_fornecedor_mes_ativo.loc[


,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes
9,1000000,crescente,4015.95,8,10,24,10,0,8.000000,7,4015.950000
10,1000000,crescente,4015.95,8,10,24,11,1,8.120000,8,4036.029750
11,1000000,crescente,4015.95,8,10,24,12,2,8.241800,13,4056.209899
12,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
13,1000000,crescente,4015.95,8,10,24,14,4,8.490908,12,4096.873403
...,...,...,...,...,...,...,...,...,...,...,...
4790,1000199,estavel,2483.11,13,3,19,15,12,13.000000,10,2483.110000
4791,1000199,estavel,2483.11,13,3,19,16,13,13.000000,22,2483.110000
4792,1000199,estavel,2483.11,13,3,19,17,14,13.000000,8,2483.110000
4793,1000199,estavel,2483.11,13,3,19,18,15,13.000000,7,2483.110000


In [9]:
sigma_valor = 0.10

mu_valor = np.log(df_fornecedor_mes_ativo['valor_medio_mes']) - (sigma_valor**2)/2

df_pgtos = df_fornecedor_mes_ativo.loc[df_fornecedor_mes_ativo.index.repeat(df_fornecedor_mes_ativo['quantidade_real_pagamentos'])].reset_index(drop=True)

display(df_pgtos[(df_pgtos['id_fornecedor_raw'] == '1000000') & (df_pgtos['mes_pagamento'] == 13)])

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes
28,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
29,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
30,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
31,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
32,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
33,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
34,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
35,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948
36,1000000,crescente,4015.95,8,10,24,13,3,8.365427,9,4076.490948


In [10]:
proporcao_ajuste = 0.3
desconto_min = 0.01 
desconto_max = 0.05 

N = len(df_pgtos)
array =  np.random.uniform(0, 1, N) # [1, 2, 3, 4, 5]
#print(array)
flag = array < proporcao_ajuste
#print(flag)
df_pgtos["flag_ajuste"] = flag

df_pgtos[(df_pgtos['id_fornecedor_raw'] == '1000000') & (df_pgtos['mes_relativo'] == 2)]

# for i in range(len(df_pgtos)): 
#     prob_ajuste = np.random.uniform(0, 1) 
#     if prob_ajuste < 0.3: 
#         df_pgtos.loc[i, "flag_ajuste"] = True 
#     else: 
#         df_pgtos.loc[i, "flag_ajuste"] = False 

# #display(df_pgtos[(df_pgtos['flag_ajuste'] == True)]) 

# display(df_pgtos)

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes,flag_ajuste
15,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,False
16,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,False
17,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,False
18,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,True
19,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,False
20,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,True
21,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,True
22,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,False
23,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,True
24,1000000,crescente,4015.95,8,10,24,12,2,8.2418,13,4056.209899,False


In [11]:
df_pgtos["mu_valor"] = np.log(df_pgtos['valor_medio_mes']) - (sigma_valor**2)/2

df_pgtos["valor_bruto"] = np.random.lognormal(df_pgtos["mu_valor"], sigma_valor)

display(df_pgtos)

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes,flag_ajuste,mu_valor,valor_bruto
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3966.097441
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4475.940332
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4101.466781
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4421.324444
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3718.020715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,1976.352737
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2192.120003
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,3005.351230
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2185.524770


In [12]:
df_pgtos["desconto"] = 0.0
mask_disk = df_pgtos['flag_ajuste'] == True
N = len(mask_disk)
desconto_min = 0.01 
desconto_max = 0.05 
desconto = np.random.uniform(desconto_min, desconto_max, N)

df_pgtos.loc[ 
    mask_disk, 
    'desconto' 
    ] = desconto[mask_disk]

df_pgtos['desconto'] = df_pgtos['valor_bruto'] * df_pgtos['desconto']

df_pgtos['valor_liquido'] = df_pgtos['valor_bruto'] - df_pgtos['desconto']
# df_pgtos[(df_pgtos['flag_ajuste'] == True)]

# df_pgtos[(df_pgtos['flag_ajuste'] == False) & (df_pgtos['desconto'] > 0)]

df_pgtos

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes,flag_ajuste,mu_valor,valor_bruto,desconto,valor_liquido
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3966.097441,0.000000,3966.097441
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4475.940332,0.000000,4475.940332
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4101.466781,0.000000,4101.466781
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,4421.324444,0.000000,4421.324444
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,4015.95,False,8.293029,3718.020715,0.000000,3718.020715
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,1976.352737,0.000000,1976.352737
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2192.120003,0.000000,2192.120003
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,3005.351230,0.000000,3005.351230
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,2483.11,False,7.812267,2185.524770,0.000000,2185.524770


In [13]:
data_base = pd.to_datetime('2024-01-01')

df_pgtos['data'] = data_base + df_pgtos['mes_pagamento'].apply(lambda x: pd.DateOffset(months=x-1))

df_pgtos['dias_no_mes'] = pd.DatetimeIndex(df_pgtos['data']).days_in_month


In [14]:
triangulo = np.random.triangular(left=0, mode=0.5, right=1, size=len(df_pgtos))
#print(triangulo)

df_pgtos['dia_pgto'] = triangulo * (df_pgtos['dias_no_mes'])

# df_pgtos['triangulo'] = df_pgtos['triangulo'] 

df_pgtos['dia_pgto'] = df_pgtos.dia_pgto.astype(int)

df_pgtos['data_transacao'] = df_pgtos['data'] + pd.to_timedelta(df_pgtos['dia_pgto'] - 1, unit='D')

mask_disk = df_pgtos['flag_ajuste'] == True

C:\Users\Adam\AppData\Local\Temp\ipykernel_20848\3375487014.py:10: PerformanceWarning: Adding/subtracting object-dtype array to TimedeltaArray not vectorized.
  df_pgtos['data_transacao'] = df_pgtos['data'] + pd.to_timedelta(df_pgtos['dia_pgto'] - 1, unit='D')


In [15]:
df_pgtos['data_transacao'] = pd.to_datetime(
    df_pgtos['data_transacao'],
    errors='coerce'
)

# mask_domingo = df_pgtos['data'] + pd.to_timedelta(df_pgtos['dia_pgto'] - 1, unit='D') == 6
mask_domingo = df_pgtos['data_transacao'].dt.dayofweek == 6
mask_sabado = df_pgtos['data_transacao'].dt.dayofweek == 5

#domingo
df_pgtos.loc[
    mask_domingo,
    'data_transacao'
] = df_pgtos.loc[mask_domingo, 'data_transacao'] + pd.Timedelta(days=1)

#sabado
df_pgtos.loc[
    mask_sabado,
    'data_transacao'
] = df_pgtos.loc[mask_sabado, 'data_transacao'] + pd.Timedelta(days=2)

df_pgtos

df_pgtos[df_pgtos['valor_liquido'] == 0]

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,valor_medio_mes,flag_ajuste,mu_valor,valor_bruto,desconto,valor_liquido,data,dias_no_mes,dia_pgto,data_transacao


In [18]:
P90 = df_pgtos['valor_bruto'].quantile(0.9)
df_pgtos['flag_alto_valor'] = df_pgtos['valor_bruto'] > P90

df_pgtos

# df_pgtos[df_pgtos['P90'] == True]

,id_fornecedor_raw,grupo,valor_medio_inicial,freq_media_inicial,mes_inicio,mes_fim,mes_pagamento,mes_relativo,lambda_mes,quantidade_real_pagamentos,...,flag_ajuste,mu_valor,valor_bruto,desconto,valor_liquido,data,dias_no_mes,dia_pgto,data_transacao,flag_alto_valor
0,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,8.293029,3966.097441,0.000000,3966.097441,2024-10-01 00:00:00,31,14,2024-10-14,False
1,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,8.293029,4475.940332,0.000000,4475.940332,2024-10-01 00:00:00,31,9,2024-10-09,False
2,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,8.293029,4101.466781,0.000000,4101.466781,2024-10-01 00:00:00,31,9,2024-10-09,False
3,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,8.293029,4421.324444,0.000000,4421.324444,2024-10-01 00:00:00,31,14,2024-10-14,False
4,1000000,crescente,4015.95,8,10,24,10,0,8.0,7,...,False,8.293029,3718.020715,0.000000,3718.020715,2024-10-01 00:00:00,31,10,2024-10-10,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39787,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,7.812267,1976.352737,0.000000,1976.352737,2025-07-01 00:00:00,31,22,2025-07-22,False
39788,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,7.812267,2192.120003,0.000000,2192.120003,2025-07-01 00:00:00,31,21,2025-07-21,False
39789,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,7.812267,3005.351230,0.000000,3005.351230,2025-07-01 00:00:00,31,18,2025-07-18,False
39790,1000199,estavel,2483.11,13,3,19,19,16,13.0,9,...,False,7.812267,2185.524770,0.000000,2185.524770,2025-07-01 00:00:00,31,26,2025-07-28,False


In [39]:
# perfil_risco

condicoes = [
    (df_pgtos['flag_ajuste'] == True) & (df_pgtos['flag_alto_valor'] == True) ,
    (df_pgtos['flag_ajuste'] == True) & (df_pgtos['flag_alto_valor'] == False),
    (df_pgtos['flag_ajuste'] == False) & (df_pgtos['flag_alto_valor'] == True),
    (df_pgtos['flag_ajuste'] == False) & (df_pgtos['flag_alto_valor'] == False)]

escolhas = ['alto_valor_retencao', 'retencao', 'alto_valor', 'normal']

df_pgtos['perfil_risco'] = np.select(condicoes, escolhas, default='Desconecido')

alto_valor = df_pgtos[df_pgtos['perfil_risco'] == 'alto_valor_retencao']['perfil_risco'].count()

In [ ]:
total = df_pgtos['perfil_risco'].count() # Porcentagem de pagamentos classificados como 'alto_valor_retencao'
alto_valor / total * 100

np.float64(3.033273019702453)